In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv('./data/titanic_procesado.csv')
df.head()


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,1.0,1.0,0.433152,0.125,0.0,0.368146,1.0
1,1,0.0,0.0,0.579431,0.125,0.0,0.615097,0.0
2,1,1.0,0.0,0.462346,0.000,0.0,0.438286,1.0
3,1,0.0,0.0,0.563806,0.125,0.0,0.595112,1.0
4,0,1.0,1.0,0.563806,0.000,0.0,0.448347,1.0


In [3]:
# Separar variables
X = df.drop(['Survived'], axis=1)
y = df['Survived']

X.head()


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1.0,1.0,0.433152,0.125,0.0,0.368146,1.0
1,0.0,0.0,0.579431,0.125,0.0,0.615097,0.0
2,1.0,0.0,0.462346,0.000,0.0,0.438286,1.0
3,0.0,0.0,0.563806,0.125,0.0,0.595112,1.0
4,1.0,1.0,0.563806,0.000,0.0,0.448347,1.0


In [4]:
y.head()


0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Convertir a arrays
X_train = X_train.values
y_train = y_train.values
X_test = X_test.values
y_test = y_test.values


In [6]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB


In [7]:
import sys
print(sys.executable)


/Users/carolinavaladezgarrido/Documents/aprendizaje_supervisado/.venv/bin/python


In [9]:
# =========================
# ENTRENAMIENTO (GridSearch)
# =========================

import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# OJO: solo si ya los instalaste y ya no te dan error
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# Definir los modelos y sus respectivos hiperparámetros para GridSearch
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear', 'saga'],
            'max_iter': [100, 500, 1000]
        }
    },
    'Clasificador de Vectores de Soporte': {
        'modelo': SVC(),
        'parametros': {
            'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'C': [0.1, 1, 10]
        }
    },
    'Clasificador de Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(),
        'parametros': {
            'splitter': ['best', 'random'],
            'max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador de Bosques Aleatorios': {
        'modelo': RandomForestClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3, 4],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'Clasificador de Gradient Boosting': {
        'modelo': GradientBoostingClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador AdaBoost': {
        'modelo': AdaBoostClassifier(),
        'parametros': {
            'n_estimators': [10, 100]
        }
    },
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(),
        'parametros': {
            'n_neighbors': [3, 5, 7]
        }
    },
    'Clasificador XGBoost': {
        'modelo': XGBClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [1, 2, 3]  # (Nota: max_depth no usa None aquí)
        }
    },
    'Clasificador LGBM': {
        'modelo': LGBMClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [-1, 1, 2, 3],  # -1 = sin límite en LightGBM
            'learning_rate': [0.1, 0.2, 0.3],
            'verbose': [-1]
        }
    },
    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros': {}
    },
    'Clasificador Naive Bayes': {
        'modelo': BernoulliNB(),
        'parametros': {
            'alpha': [0.1, 1.0, 10.0]
        }
    }
}

# Inicializar variables para almacenar los puntajes de los modelos y el mejor estimador
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():
    grid_search = GridSearchCV(
        estimator=info_modelo['modelo'],
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=-1,
    )

    # Ajustar GridSearchCV con los datos de entrenamiento
    grid_search.fit(X_train, y_train)

    # Hacer predicciones con el modelo ajustado
    y_pred = grid_search.predict(X_test)

    # Calcular la precisión de las predicciones
    precision = accuracy_score(y_test, y_pred)

    # Almacenar los resultados del modelo
    puntajes_modelos.append({
        'Modelo': nombre,
        'Precisión': precision
    })

    # Guardar mejor estimador por modelo
    estimadores[nombre] = grid_search.best_estimator_

    # Actualizar el mejor modelo global
    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

# Convertir los resultados a un DataFrame para una mejor visualización
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))

print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")


/Users/carolinavaladezgarrido/Documents/aprendizaje_supervisado/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/carolinavaladezgarrido/Documents/aprendizaje_supervisado/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/carolinavaladezgarrido/Documents/aprendizaje_supervisado/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/carolinavaladezgarrido/Documents/aprendizaje_supervisado/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Use

Rendimiento de los modelos de clasificación
                                 Modelo  Precisión
4     Clasificador de Gradient Boosting       0.82
6      Clasificador K-Nearest Neighbors       0.82
7                  Clasificador XGBoost       0.82
8                     Clasificador LGBM       0.82
1   Clasificador de Vectores de Soporte       0.80
2     Clasificador de Árbol de Decisión       0.80
3    Clasificador de Bosques Aleatorios       0.80
0                   Regresión Logística       0.79
5                 Clasificador AdaBoost       0.79
10             Clasificador Naive Bayes       0.79
9                            GaussianNB       0.77
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador de Gradient Boosting
Precisión: 0.82


In [10]:
print("Mejor modelo:", mejor_modelo)
print("Mejor precisión:", mejor_precision)
print("Tipo de mejor estimador:", type(mejor_estimador))


Mejor modelo: Clasificador de Gradient Boosting
Mejor precisión: 0.8156424581005587
Tipo de mejor estimador: <class 'sklearn.ensemble._gb.GradientBoostingClassifier'>


In [11]:
# Esto ya lo tenemos importado. Lo ponemos nuevamente nada más de referencia
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# Creamos el modelo de regresión logística
model = LogisticRegression()

# Entrenamos el modelo con los datos de entrenamiento
model.fit(X_train, y_train)

# Realizamos predicciones con el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluamos el modelo usando precisión
accuracy = accuracy_score(y_test, y_pred)

print(f"Precisión del modelo: {accuracy:.2f}")

Precisión del modelo: 0.80


In [12]:
model = LogisticRegression(
    C=0.5,                  # Valor de regularización
    penalty='l2',            # Tipo de penalización (l2 es la regularización Ridge)
    solver='lbfgs',         # Algoritmo de optimización
    max_iter=200,           # Número máximo de iteraciones
    class_weight='balanced' # Ajustar pesos de las clases
)

# Entrenar el modelo
model.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluar la precisión del modelo
accuracy = accuracy_score(y_test, y_pred)

print(f"Precisión del modelo: {accuracy:.2f}")

Precisión del modelo: 0.77


In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# 1) Cargar datos
df = pd.read_csv('./data/titanic_procesado.csv')

# 2) Separar X e y
X = df.drop(['Survived'], axis=1)
y = df['Survived']

# 3) Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# (opcional) convertir a arrays
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values


# 4) Diccionario de modelos
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear', 'saga'],
            'max_iter': [100, 500, 1000]
        }
    },
    'SVC': {
        'modelo': SVC(),
        'parametros': {
            'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'C': [0.1, 1, 10]
        }
    },
    'Decision Tree': {
        'modelo': DecisionTreeClassifier(),
        'parametros': {
            'splitter': ['best', 'random'],
            'max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Random Forest': {
        'modelo': RandomForestClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3, 4],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'Gradient Boosting': {
        'modelo': GradientBoostingClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3, 4]
        }
    },
    'AdaBoost': {
        'modelo': AdaBoostClassifier(),
        'parametros': {
            'n_estimators': [10, 100]
        }
    },
    'KNN': {
        'modelo': KNeighborsClassifier(),
        'parametros': {
            'n_neighbors': [3, 5, 7]
        }
    },
    'XGBoost': {
        'modelo': XGBClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [1, 2, 3]
        }
    },
    'LGBM': {
        'modelo': LGBMClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3],
            'learning_rate': [0.1, 0.2, 0.3],
            'verbose': [-1]
        }
    },
    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros': {}
    },
    'BernoulliNB': {
        'modelo': BernoulliNB(),
        'parametros': {
            'alpha': [0.1, 1.0, 10.0]
        }
    }
}

# 5) Entrenamiento + búsqueda
puntajes_modelos = []
mejor_precision = -1
mejor_estimador = None
mejor_modelo = None
estimadores = {}

for nombre, info in modelos.items():
    grid_search = GridSearchCV(
        estimator=info['modelo'],
        param_grid=info['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)

    y_pred = grid_search.predict(X_test)
    precision = accuracy_score(y_test, y_pred)

    puntajes_modelos.append({'Modelo': nombre, 'Precisión': precision})
    estimadores[nombre] = grid_search.best_estimator_

    if precision > mejor_precision:
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_
        mejor_modelo = nombre

metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))
print('--------------------------')
print("MEJOR MODELO")
print("Modelo:", mejor_modelo)
print("Precisión:", round(mejor_precision, 2))


/Users/carolinavaladezgarrido/Documents/aprendizaje_supervisado/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/carolinavaladezgarrido/Documents/aprendizaje_supervisado/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/carolinavaladezgarrido/Documents/aprendizaje_supervisado/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/carolinavaladezgarrido/Documents/aprendizaje_supervisado/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Use

Rendimiento de los modelos de clasificación
                 Modelo  Precisión
4     Gradient Boosting       0.82
6                   KNN       0.82
7               XGBoost       0.82
8                  LGBM       0.82
1                   SVC       0.80
2         Decision Tree       0.80
3         Random Forest       0.80
0   Regresión Logística       0.79
5              AdaBoost       0.79
10          BernoulliNB       0.79
9            GaussianNB       0.77
--------------------------
MEJOR MODELO
Modelo: Gradient Boosting
Precisión: 0.82


In [27]:
nuevos_datos = np.array(X_train[0]).reshape(1, -1)
prediccion = mejor_estimador.predict(nuevos_datos)
print("Predicción:", prediccion)


Predicción: [0]


In [28]:
print(mejor_modelo)
print(mejor_precision)
print(mejor_estimador)


Gradient Boosting
0.8156424581005587
GradientBoostingClassifier(max_depth=2)


In [29]:
nuevos_datos = np.array(X_train[0]).reshape(1, -1)
print("Predicción:", mejor_estimador.predict(nuevos_datos))
print("Valor real:", y_train[0])


Predicción: [0]
Valor real: 0


In [30]:
import os
os.path.exists("modelo.pkl")


False

In [31]:
import os
print(os.getcwd())
print(os.listdir())


/Users/carolinavaladezgarrido/Documents/aprendizaje_supervisado
['4_machine_learning.ipynb', '1_clean.ipynb', '5_pipeline.ipynb', 'LICENSE', 'requirements.txt', 'README.md', '2_EDA.ipynb', '.gitignore', '3_feature_engineering.ipynb', '.venv', 'app.py', '.ipynb_checkpoints', '.git', 'data']


In [32]:
print(mejor_estimador)


GradientBoostingClassifier(max_depth=2)


In [33]:
import pickle

with open("modelo.pkl", "wb") as archivo:
    pickle.dump(mejor_estimador, archivo)

print("Modelo guardado correctamente")


Modelo guardado correctamente


In [34]:
import os
print(os.listdir())


['4_machine_learning.ipynb', '1_clean.ipynb', '5_pipeline.ipynb', 'LICENSE', 'requirements.txt', 'README.md', '2_EDA.ipynb', '.gitignore', '3_feature_engineering.ipynb', '.venv', 'app.py', '.ipynb_checkpoints', '.git', 'modelo.pkl', 'data']
